<a href="https://colab.research.google.com/github/KatiaItzelCortes/Simulacion-1/blob/main/Linea%20de%20espera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sistema de línea de espera con un servidor

Considere una estación de servicio a la cual los clientes llegan de acuerdo con un proceso Poisson no homogéneo con función de intensidad $\lambda(t)$, $\lambda \geq 0$. Hay un único servidor, y al llegar un cliente pasa a servicio si el servidor está libre en ese momento, o bien se une a la fila de espera si está ocupado. Cuando el servidor termina de dar servicio a un cliente, se ocupa del cliente que ha estado esperando más tiempo (la disciplina "primero en llegar, primero en atender") si hay clientes esperando, o bien, si no los hay, permanece libre hasta la llegada del siguiente cliente.

Para simular el sistema anterior utilizamos las siguientes variables:

1. **Variable de tiempo:** $t$
2. **Variables de conteo:**
    * $N_A$: el número de llegadas (hasta el instante $t$)
   * $N_D$: el número de salidas (hasta el instante $t$)
3. **Variable de estado del sistema:**
   * $n$: el número de clientes en el sistema (en el instante $t$)


Para comenzar la simulación, inicializamos las variables y los tiempos de los eventos como sigue:

**Inicialización**
* Sean $t = N_A = N_D = 0$.
* Sea $ES = 0$.
* Generar $T_0$, y hacer $t_A = T_0, t_D = \infty$.
$$t = \text{variable de tiempo}, \quad ES = n, \quad LE = t_A, t_D$$


**Caso 1:** $t_A \leq t_D, \quad t_A \leq T$
* **Restablecer:** $t = t_A$ (nos movemos hasta el tiempo $t_A$).
* **Restablecer:** $N_A = N_A + 1$ (pues hay una llegada adicional en el instante $t_A$).
* **Restablecer:** $n = n + 1$ (pues ahora se tiene un cliente más).
* **Generar** $T_t$, y hacer $t_A = T_t$ (ésta es la hora de la siguiente llegada).
* **Si** $n = 1$, generar $Y$ y hacer $t_D = t + Y$ (pues el sistema ha quedado vacío, por lo cual necesitamos generar el tiempo de servicio del nuevo cliente).
* **Reunir** los datos de salida $A(N_A) = t$ (pues el cliente $N_A$ llega en el instante $t$).

**Caso 2:** $t_D < t_A, \quad t_D \leq T$
* **Restablecer:** $t = t_D$.
* **Restablecer:** $n = n - 1$.
* **Restablecer:** $N_D = N_D + 1$ (pues ha ocurrido una salida en el instante $t$).
* **Si** $n = 0$, hacer $t_D = \infty$; en caso contrario, generar $Y$ y hacer $t_D = t + Y$.
* **Reunir** los datos de salida $D(N_D) = t$ (pues el cliente $N_D$ acaba de salir).

**Caso 3:** $\min(t_A, t_D) > T, \quad n > 0$
* **Restablecer:** $t = t_D$.
* **Restablecer:** $n = n - 1$.
* **Restablecer:** $N_D = N_D + 1$.
* **Si** $n > 0$, generar $Y$ y hacer $t_D = t + Y$.
* **Reunir** los datos de salida $D(N_D) = t$.

**Caso 4:** $\min(t_A, t_D) > T, \quad n = 0$
* **Reunir** los datos de salida $T_p = \max(t - T, 0)$.



In [1]:
import random as r
import numpy as np
import matplotlib.pyplot as plt

In [25]:
def sdldecus(T, lam, mu):
    t = 0.0
    Na = 0
    Nd = 0
    n = 0

    A = {}
    D = {}
    def ti_llegada(t_actual, tasa_lam):
        R1 = r.random()
        return t_actual - (1 / tasa_lam) * np.log(R1)

    def ti_servicio(tasa_mu):
        R2 = r.random()
        return -(1 / tasa_mu) * np.log(R2)

    ta = ti_llegada(t, lam)
    td = float('inf')

    while True:
        # Caso 1
        if ta <= td and ta <= T:
            t = ta
            Na += 1
            n += 1
            ta = ti_llegada(t, lam)
            if n == 1:
                Y = ti_servicio(mu)
                td = t + Y
            A[Na] = t

        # Caso 2
        elif td < ta and td <= T:
            t = td
            n -= 1
            Nd += 1

            if n == 0:
                td = float('inf')
            else:
                Y = ti_servicio(mu)
                td = t + Y
            D[Nd] = t

        # Caso 3
        elif min(ta, td) > T and n > 0:
            t = td
            n -= 1
            Nd += 1

            if n > 0:
                Y = ti_servicio(mu)
                td = t + Y
            D[Nd] = t

        # Caso 4
        elif min(ta, td) > T and n == 0:
            Tp = max(t - T, 0)
            break

    # Cálculo final
    ti_sistema = [D[i] - A[i] for i in range(1, Na + 1)]

    tiempo_promedio_sistema = np.mean(ti_sistema) if ti_sistema else 0

    return {
        "Total_Llegadas": Na,
        "Total_Salidas": Nd,
        "Tiempo_promedio_sistema_Simulado": tiempo_promedio_sistema,
        "Tiempo_extra_servidor_Tp": Tp
    }

In [28]:
# Parámetros solicitados
lam = 4.0      # Tasa de llegada
mu = 6.0       # Tasa de servicio
T_simulacion = 10000.0

resultados_sim = sdldecus(T_simulacion, lam, mu)
print(f"Na = {resultados_sim['Total_Llegadas']}")
print(f"Tp = {resultados_sim['Tiempo_extra_servidor_Tp']:}")
print(f"Tiempo promedio simulado = {resultados_sim['Tiempo_promedio_sistema_Simulado']}")
print(f"Total de clientes atendidos: {resultados_sim['Total_Salidas']}")

if mu > lam:
    W = 1 / (mu - lam)
    print(f"Tiempo promedio en el sistema: {W}")
else:
    print("El sistema es inestable matemáticamente (lambda >= mu).")

Na = 39985
Tp = 1.2495035052088497
Tiempo promedio simulado = 0.4851864488859644
Total de clientes atendidos: 39985
Tiempo promedio en el sistema: 0.5
